# 03 — Entraînement LoRA
1) Première LoRA simple (rang 8). 2) Balayage des rangs 4/8/16/32 avec checkpoints 250/500/1000/1500.

In [ ]:
# Setup — works on Colab / Kaggle AND locally (VS Code, Jupyter)
import os, sys, subprocess
from pathlib import Path

ON_CLOUD = "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if ON_CLOUD:
    if not Path("lora-zellige").exists():
        subprocess.run(["git", "clone", "https://github.com/braziyass/lora-zellige.git"], check=True)
    os.chdir("lora-zellige")
else:
    # Locally the notebook starts in notebooks/ : go up to the repo root
    root = Path.cwd().resolve()
    while not (root / "requirements.txt").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)
sys.path.insert(0, os.getcwd())
PY = f'"{sys.executable}"'   # used as !{PY} -m ... in the cells below
print("Working directory:", os.getcwd())

print("Python used by this notebook:", sys.executable)

# Packages: installed automatically on Colab/Kaggle only.
# Locally, install once in the Terminal (see README), or set INSTALL = True.
INSTALL = ON_CLOUD
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for notebooks 01-02 data steps)")
except ImportError:
    print("torch not installed: only needed from notebook 02 on (use Colab/Kaggle for training)")

In [ ]:
# Optionnel : persister données et LoRA sur Google Drive entre les sessions
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/lora-zellige/{raw,outputs}
# !ln -sfn /content/drive/MyDrive/lora-zellige/raw data/raw
# !ln -sfn /content/drive/MyDrive/lora-zellige/outputs outputs

## 1. Première LoRA

In [ ]:
!{PY} -m src.train_lora --config configs/base.yaml --set experiment_name=r8

## 2. Balayage des rangs (~1 h sur T4 ; reprise automatique si la session coupe)

In [ ]:
!{PY} -m src.sweep --sweep configs/sweep.yaml

## 3. Courbes de perte et coûts

In [ ]:
import json, glob, pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
rows = []
for d in sorted(glob.glob("outputs/loras/*/")):
    log = pd.read_csv(d + "train_log.csv")
    ax.plot(log.step, log.loss.rolling(10, min_periods=1).mean(), label=d.split("/")[-2])
    info = json.load(open(d + "run_info.json"))
    rows.append({k: info[k] for k in ["experiment", "train_time_min", "peak_vram_gb", "trainable_params", "trainable_pct_unet"]}
                | {"lora_size_mb": max(info["lora_size_mb"].values())})
ax.set_xlabel("étape"); ax.set_ylabel("perte (moyenne glissante)"); ax.legend()
plt.savefig("results/figures/loss.png", dpi=150)
costs = pd.DataFrame(rows); costs.to_csv("results/costs.csv", index=False); costs

La perte de diffusion est très bruitée et **ne suffit pas** à détecter le surapprentissage → voir notebook 04.